# StealthOS — Hybrid ML Malware Detection Training Notebook
This notebook trains the machine learning models used by the StealthOS threat intelligence backend:
1. **XGBoost Classifier** — High-accuracy supervised classifier trained on EMBER2024 PE features.
2. **Random Forest Classifier** — Ensemble model for robust threat label prediction.
3. **PyTorch Autoencoder** — Unsupervised anomaly detection network designed to identify novel/zero-day threats based on high reconstruction MSE.

Models are evaluated, analyzed via SHAP, and exported to `.pkl` and `.pt` formats to be copied into the backend `models/` directory.

## 1. Setup and Environment Initialization
Install required dependencies for training, visualization, and explainability.

In [ ]:
# Install dependencies if running on Google Colab
!pip install -q xgboost shap scikit-learn torch matplotlib pandas numpy

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import shap

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("All libraries imported successfully.")

## 2. Dataset Synthesis (EMBER2024 PE Feature Representation)
To emulate the EMBER2024 feature format, we generate a synthetic dataset of 64-dimensional feature vectors representing benign and malicious executables. Features correspond to standard binary characteristics (entropy, import count, sections, API counts, packers, etc.).

In [ ]:
def generate_synthetic_ember_data(n_samples=2000):
    # 64 features. Column descriptions:
    # 0: overall_entropy, 1: max_section_entropy, 2: overlay_entropy, 3: dangerous_imports_count,
    # 4: suspicious_str_count, 5: urls_count, 6: ips_count, 7: is_packed (0/1), ...
    X = np.zeros((n_samples, 64), dtype=np.float32)
    y = np.zeros(n_samples, dtype=np.int32) # Labels: 0=Clean, 1=Adware, 2=Spyware, 3=Trojan, 4=Ransomware
    
    for i in range(n_samples):
        label = np.random.choice([0, 1, 2, 3, 4], p=[0.5, 0.1, 0.15, 0.15, 0.1])
        y[i] = label
        
        if label == 0:  # Clean
            X[i, 0] = np.random.uniform(2.0, 5.0)  # Low entropy
            X[i, 1] = np.random.uniform(2.5, 5.5)
            X[i, 2] = 0.0                          # No overlay
            X[i, 3] = np.random.randint(0, 2)      # Few/no dangerous imports
            X[i, 4] = np.random.randint(0, 5)      # Few suspicious strings
            X[i, 7] = 0.0                          # Not packed
            # Fill normal imports
            X[i, 12] = np.random.randint(5, 30)
        
        elif label == 1: # Adware
            X[i, 0] = np.random.uniform(4.0, 6.0)
            X[i, 1] = np.random.uniform(4.5, 6.5)
            X[i, 3] = np.random.randint(0, 3)
            X[i, 5] = np.random.randint(5, 20)     # Many URLs
            X[i, 6] = np.random.randint(2, 8)      # IPs
            X[i, 23] = np.random.uniform(0.6, 1.0) # Adware strings match
            X[i, 12] = np.random.randint(10, 40)

        elif label == 2: # Spyware
            X[i, 0] = np.random.uniform(4.5, 6.5)
            X[i, 1] = np.random.uniform(5.0, 7.0)
            X[i, 3] = np.random.randint(2, 8)      # Spyware APIs (clipboard, keylog)
            X[i, 4] = np.random.randint(5, 15)
            X[i, 6] = np.random.randint(1, 5)
            X[i, 17] = np.random.uniform(0.7, 1.0) # Spyware imports
            X[i, 22] = np.random.uniform(0.8, 1.0) # Spyware strings
            X[i, 12] = np.random.randint(15, 50)

        elif label == 3: # Trojan
            X[i, 0] = np.random.uniform(5.5, 7.5)  # Elevated entropy
            X[i, 1] = np.random.uniform(6.0, 7.8)
            X[i, 3] = np.random.randint(4, 12)     # Trojan API hooks (injection)
            X[i, 7] = 1.0                          # Often packed
            X[i, 11] = np.random.randint(1, 4)     # Writable + Executable sections
            X[i, 16] = np.random.uniform(0.8, 1.0) # Trojan imports
            X[i, 19] = np.random.uniform(0.6, 1.0) # Evasion imports
            X[i, 12] = np.random.randint(15, 60)

        elif label == 4: # Ransomware
            X[i, 0] = np.random.uniform(6.2, 7.9)  # High entropy
            X[i, 1] = np.random.uniform(6.5, 7.95)
            X[i, 2] = np.random.uniform(5.5, 7.9)  # Encrypted overlay
            X[i, 3] = np.random.randint(3, 10)     # Crypto APIs
            X[i, 4] = np.random.randint(8, 25)     # Ransom vocabulary
            X[i, 15] = np.random.uniform(0.8, 1.0) # Ransomware imports
            X[i, 20] = np.random.uniform(0.7, 1.0) # Ransomware strings
            X[i, 29] = 1.0                         # High entropy flag
            X[i, 12] = np.random.randint(10, 45)
            
        # Add normal noise across remaining feature dimensions (index 32-63 represent dynamic sandbox behavior)
        if label > 0:
            X[i, 32:64] = np.random.uniform(0.3, 0.95, size=32)
        else:
            X[i, 32:64] = np.random.uniform(0.0, 0.25, size=32)

    return X, y

X, y = generate_synthetic_ember_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dataset synthesized. Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 3. Supervised Model 1: Random Forest Classifier
Train a classic Random Forest model for multi-class classification.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
print("Random Forest Evaluation:")
print(classification_report(y_test, rf_preds, target_names=['Clean', 'Adware', 'Spyware', 'Trojan', 'Ransomware']))

## 4. Supervised Model 2: XGBoost Classifier
Train an XGBoost model using gradient boosting trees.

In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=120,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    random_state=42
)
xgb_clf.fit(X_train, y_train)

xgb_preds = xgb_clf.predict(X_test)
print("XGBoost Evaluation:")
print(classification_report(y_test, xgb_preds, target_names=['Clean', 'Adware', 'Spyware', 'Trojan', 'Ransomware']))

## 5. Unsupervised Model: PyTorch Autoencoder (Zero-Day Anomaly Detection)
Train an autoencoder on **benign data only** (label=0). During prediction, if reconstruction Mean Squared Error (MSE) is above a configured threshold, the executable is flagged as a potential anomaly / Zero-Day threat.

In [ ]:
# Filter only clean samples for autoencoder training
X_train_clean = X_train[y_train == 0]
X_test_clean = X_test[y_test == 0]
X_test_malicious = X_test[y_test > 0]

# PyTorch dataset loaders
train_dataset = TensorDataset(torch.tensor(X_train_clean, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16), # Bottleneck layer
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64)
        )
        
    def forward(self, x):
        return self.decoder(self.encoder(x))

ae = Autoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(ae.parameters(), lr=0.005)

# Train loop
epochs = 30
ae.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch in train_loader:
        x_batch = batch[0]
        optimizer.zero_grad()
        outputs = ae(x_batch)
        loss = criterion(outputs, x_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x_batch.size(0)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{epochs} | MSE Loss: {epoch_loss / len(X_train_clean):.6f}")

### Autoencoder Threshold Optimization

In [ ]:
ae.eval()
with torch.no_grad():
    # Compute reconstruction errors for clean samples
    clean_tensors = torch.tensor(X_test_clean, dtype=torch.float32)
    clean_reconstructions = ae(clean_tensors)
    clean_errors = torch.mean((clean_tensors - clean_reconstructions) ** 2, dim=1).numpy()
    
    # Compute reconstruction errors for malicious samples
    mal_tensors = torch.tensor(X_test_malicious, dtype=torch.float32)
    mal_reconstructions = ae(mal_tensors)
    mal_errors = torch.mean((mal_tensors - mal_reconstructions) ** 2, dim=1).numpy()

# Select a threshold (e.g. 95th percentile of clean errors)
threshold = np.percentile(clean_errors, 95)
print(f"95th Percentile Clean Error Threshold: {threshold:.6f}")

false_positives = np.sum(clean_errors > threshold) / len(clean_errors)
true_positives = np.sum(mal_errors > threshold) / len(mal_errors)
print(f"False Positive Rate (Benign flagged anomaly): {false_positives*100:.1f}%")
print(f"True Positive Rate (Malware flagged anomaly): {true_positives*100:.1f}%")

## 6. Model Explainability with SHAP
SHAP (SHapley Additive exPlanations) values help explain feature contribution. We plot the summary of our XGBoost model.

In [ ]:
# Explain model predictions using SHAP TreeExplainer
explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test)

# Map column indexes to meaningful PE features
feature_names = [f"Feature_{i}" for i in range(64)]
feature_names[0] = "overall_entropy"
feature_names[1] = "max_section_entropy"
feature_names[3] = "dangerous_imports_count"
feature_names[4] = "suspicious_str_count"
feature_names[7] = "is_packed"
feature_names[12] = "total_imports"

# Plot summary of feature importance
shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_size=(10, 6))

## 7. Model Serialization & Export
Export all models to disk. These binaries can be loaded by `backend/classifier.py` for inference.

In [ ]:
# Create models folder if not existing
os.makedirs("../backend/models", exist_ok=True)

# Export Random Forest
rf_out_path = "../backend/models/rf_model.pkl"
with open(rf_out_path, "wb") as f:
    pickle.dump(rf, f)
print(f"Random Forest exported to {rf_out_path}")

# Export XGBoost
xgb_out_path = "../backend/models/xgboost_model.pkl"
with open(xgb_out_path, "wb") as f:
    pickle.dump(xgb_clf, f)
print(f"XGBoost exported to {xgb_out_path}")

# Export PyTorch Autoencoder state dict / model
ae_out_path = "../backend/models/autoencoder.pt"
torch.save(ae, ae_out_path)
print(f"PyTorch Autoencoder exported to {ae_out_path}")

print("\nAll models successfully serialized and placed in backend/models/")